In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

### 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [3]:
df = pd.read_csv('data-logistic.csv', header=None)
y = df.iloc[:, 0].values.astype(float)
X = df.iloc[:, 1:3].values.astype(float) 

### 2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!

### 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

### 4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.

In [4]:
def train_logistic(C, k=0.1, w_init=None, max_iter=10000, tol=1e-5):
    w = np.zeros(2) if w_init is None else w_init.copy()
    converged = False
    for i in range(1, max_iter + 1):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(np.clip(-y * z, -700, 700)))
        grad = X.T @ (y * (1.0 - p)) / len(y)
        w_new = w + k * grad - k * C * w
        if np.linalg.norm(w_new - w) <= tol:
            w = w_new
            converged = True
            return w, converged, i
        w = w_new
    return w, converged, max_iter

In [5]:
w0, _, iters0 = train_logistic(0.0)
w10, _, iters10 = train_logistic(10.0)

### 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом.Для этого воспользуйтесь сигмоидной функцией: a(x) = 1/(1 + exp(−w1 x1 − w2 x2 )).

In [6]:
def get_proba(w):
    return 1.0 / (1.0 + np.exp(np.clip(-X @ w, -700, 700)))

In [11]:
auc0 = roc_auc_score(y, get_proba(w0))
auc10 = roc_auc_score(y, get_proba(w10))
print(f"П.5: AUC (C=0)={auc0:.3f} | AUC (C=10)={auc10:.3f}")

П.5: AUC (C=0)=0.927 | AUC (C=10)=0.936


### 6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?

In [16]:
for test_k in [1.0, 0.1, 0.01]:
    _, conv, it = train_logistic(0.0, k=test_k)
    print(f"k={test_k:4.2f} | Сходится: {'' if conv else ''} | Итераций: {it}")

k=1.00 | Сходится:  | Итераций: 32
k=0.10 | Сходится:  | Итераций: 244
k=0.01 | Сходится:  | Итераций: 1479


### 7. Попробуйте менять начальное приближение. Влияет ли оно на что-нибудь?

In [17]:
w_base, _, it_base = train_logistic(0.0, k=0.1, w_init=np.array([0.0, 0.0]))
w_alt, _, it_alt   = train_logistic(0.0, k=0.1, w_init=np.array([10.0, -5.0]))

print(f"Старт (0,0)   → AUC: {roc_auc_score(y, get_proba(w_base)):.3f}, итераций: {it_base}")
print(f"Старт (10,-5) → AUC: {roc_auc_score(y, get_proba(w_alt)):.3f}, итераций: {it_alt}")
print(f"Разница финальных весов: {np.linalg.norm(w_base - w_alt):.2e}")

Старт (0,0)   → AUC: 0.927, итераций: 244
Старт (10,-5) → AUC: 0.927, итераций: 656
Разница финальных весов: 8.09e-04
